In [98]:
from google.colab import drive # remove the cell if not using colab
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [99]:
import pandas as pd
import numpy as np
from pathlib import Path
base_path = Path('/content/drive/MyDrive/solvro_dane') # change path here!

# Klasyfikacja pasażerów Titanica
Nie wiemy czy dla DiCaprio było miejsce na drzwiach, ale wiemy że grdyby był tam Wojfer87 to by z nimi wyciskał pompki na górze lodowej. Teraz twoja pora na wyciskanie.
#Twoje zadnie to:
**stworzenie modelu przewidującego szanse przeżycia katastrofy Titanica**.

![https://i1.jbzd.com.pl/contents/2025/11/normal/v5Fth4DcPpPPxSrXQ5rCbAgZ8EifWiiF.png](https://i1.jbzd.com.pl/contents/2025/11/normal/v5Fth4DcPpPPxSrXQ5rCbAgZ8EifWiiF.png "Wojfer")



#### Twoim celem będzie jest wytrenowanie modeli do klasyfikacji każdego pasażera Titanica jako ofiary (0) lub osoby, która przeżyła (1).

Poniżej znajdziesz pytania, które mogą być pomocne w zadaniu:

- Czego nauczyło Cię o badanym zbiorze danych poprzednie zadanie? Jak możesz wykorzystać wyciągnięte z niego wnioski w procesie tworzenia modelu?
- Jak przeprowadzenie standaryzacji danych może wpływać na zachowanie modelu?
- Co mój model robi i w jaki sposób?
- Jak nie przetrenować wybranego modelu?
- Jaki wynik klasyfikacji możemy uznać za *dobry*?


Wymagania:
- Wypisz obserwacje z pierwszego zadania, które pomogą Ci w tym. Co było przydatne, a co okazało się bezużyteczne?
- [Nie doprowadź](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) do ~~przecieku statku~~ wycieku danych (np. nie ucz modelu na danych testowych). Nauczone modele odpal na danych treningowych i testowych - opisz uzyskane wyniki.
- Stwórz baseline, czyli dla porównania sprawdź jak z zadaniem radzi sobie [Dummy Classifier](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html) (jeśli Twój docelowy model radzi sobie gorzej - uciekaj)
- Przeprowadź badania na dwóch wybranych modelach uczenia maszynowego (np. spośród: drzew decyzyjnych, SVM, MLP, KNN, z gwiazdką [XGBoost](https://xgboost.readthedocs.io/en/stable))
- W badaniach użyj wybranych metryk. Wybór uzasadnij.
- Dla każdego modelu wybierz co najmniej dwa hiperparametry i przeprowadź badania zależności wyników metryk od wartości hiperparametrów. Zwizualizuj wszystko ładnie, zastanów się dlaczego tak mogło być i wyciągnij i wypisz wnioski.
- Podsumuj przeprowadzone badania, wypisz wnioski.

Niezmiennie, zadbaj o czytelność kodu i nazewnictwo zmiennych. Jeśli jakiś wycinek kodu się powtarza, to wyodrębnij go do funkcji. Postaraj się zamieszczać swoje wnioski w postaci komentarza `Markdown`.

Jeśli chcesz, możesz sprawdzić (przyjmując pewne założenia), jakie byłyby Twoje szanse na Titanicu.

Uwaga! Jeśli Titanic to dla Ciebie nic i baaaaardzo chcesz to możesz w ramach tego zadania zająć się [bardziej wymagającym](https://archive.ics.uci.edu/dataset/365/polish+companies+bankruptcy+data) zbiorem.

In [100]:
titanic_df = pd.read_csv(base_path / 'titanic_final.csv', index_col='PassengerId')

# Wnioski z EDA
Większą szansę na przeżycie miały:
* kobiety
* dzieci
* pasażerowie wyższych klas
* pasażerowie z przypisanymi kabinami (łączy się z klasą)

# Ładowanie i dzielenie danych

In [101]:
from sklearn.model_selection import train_test_split

X = titanic_df.drop(columns=['Survived'])
y = titanic_df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=.8, shuffle=True)

# Skalowanie danych


In [102]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Baseline - Dummy Classifier

In [103]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, classification_report
from sklearn.dummy import DummyClassifier

model = DummyClassifier(strategy='most_frequent') # dane nie są zbalansowane - większość zginęła
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print(classification_report(y_test, y_pred))
confusion_matrix(y_test, y_pred)

              precision    recall  f1-score   support

           0       0.60      1.00      0.75       107
           1       0.00      0.00      0.00        72

    accuracy                           0.60       179
   macro avg       0.30      0.50      0.37       179
weighted avg       0.36      0.60      0.45       179



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



array([[107,   0],
       [ 72,   0]])

# Support Vector Machine

In [104]:
from sklearn.svm import SVC

svm = SVC(kernel='linear')  # liniowe SVM
svm.fit(X_train_scaled, y_train)

y_pred = svm.predict(X_test_scaled)

print(classification_report(y_test, y_pred))
confusion_matrix(y_test, y_pred)

              precision    recall  f1-score   support

           0       0.81      0.91      0.85       107
           1       0.83      0.68      0.75        72

    accuracy                           0.82       179
   macro avg       0.82      0.79      0.80       179
weighted avg       0.82      0.82      0.81       179



array([[97, 10],
       [23, 49]])

# Multi Layer Perceptron

In [105]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(max_iter=1000, random_state=42, hidden_layer_sizes=(64, 32), early_stopping=True)
mlp.fit(X_train_scaled, y_train)

y_pred = mlp.predict(X_test_scaled)

print(classification_report(y_test, y_pred))
confusion_matrix(y_test, y_pred)

              precision    recall  f1-score   support

           0       0.78      0.94      0.86       107
           1       0.88      0.61      0.72        72

    accuracy                           0.81       179
   macro avg       0.83      0.78      0.79       179
weighted avg       0.82      0.81      0.80       179



array([[101,   6],
       [ 28,  44]])

# K Nearest Neighbors


In [106]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=10, metric='cosine')
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

print(classification_report(y_test, y_pred))
confusion_matrix(y_test, y_pred)

              precision    recall  f1-score   support

           0       0.77      0.95      0.85       107
           1       0.89      0.57      0.69        72

    accuracy                           0.80       179
   macro avg       0.83      0.76      0.77       179
weighted avg       0.82      0.80      0.79       179



array([[102,   5],
       [ 31,  41]])

W zależności od tego jak załadują się dane, modele dają znacząco różne wyniki.

# Cross-Validation

In [107]:
from sklearn.model_selection import cross_val_score

svm_score = cross_val_score(svm, X, y, cv=5, scoring='f1')
mlp_score = cross_val_score(mlp, X, y, cv=5, scoring='f1')
knn_score = cross_val_score(knn, X, y, cv=5, scoring='f1')

print('---------- SVM ----------')
print(svm_score)
print(f"srednia: {svm_score.mean():.3f}  odchylenie: {svm_score.std():.3f}")

print('\n---------- MLP ----------')
print(mlp_score)
print(f"srednia: {mlp_score.mean():.3f}  odchylenie: {mlp_score.std():.3f}")

print('\n---------- KNN ----------')
print(knn_score)
print(f"srednia: {knn_score.mean():.3f}  odchylenie: {knn_score.std():.3f}")

---------- SVM ----------
[0.7826087  0.76691729 0.75384615 0.70588235 0.82089552]
srednia: 0.766  odchylenie: 0.038

---------- MLP ----------
[0.38655462 0.55932203 0.50485437 0.59459459 0.56198347]
srednia: 0.521  odchylenie: 0.073

---------- KNN ----------
[0.58015267 0.64       0.59016393 0.61016949 0.60714286]
srednia: 0.606  odchylenie: 0.020


Wybrałem metrykę f1 - jest ona harmoniczną średnią z precision i recall. Model musi dobrze wybierać, ale też nie może się za często mylić. Z obecnymi parametrami dość często się myli.

In [108]:
!pip install optuna
import optuna
from sklearn.pipeline import Pipeline
import optuna.visualization as vis

# Szukanie hiperparametrów dla SVM

In [109]:
def svm_search(trial):
  c_param = trial.suggest_float('C', 0.01, 100, log=True)

  kernel_param = trial.suggest_categorical('kernel', ['linear', 'rbf'])

  if kernel_param == 'rbf':
      gamma_param = trial.suggest_float('gamma', 0.0001, 10, log=True)
  else:
      gamma_param = 'scale'

  class_weight_param = trial.suggest_categorical('class_weight', [None, 'balanced'])

  svm = SVC(C=c_param, kernel=kernel_param, gamma=gamma_param,
            class_weight=class_weight_param, random_state=42)

  pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', svm)
    ])

  score = cross_val_score(pipeline, X, y, cv=5, scoring='f1').mean()
  return score

study = optuna.create_study(direction='maximize')

study.optimize(svm_search, n_trials=100)

print("\n--- ZAKOŃCZONO ---")
print(f"Najlepsze Accuracy: {study.best_value:.4f}")
print("Najlepsze parametry:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")


[I 2026-08-15 21:57:43,085] A new study created in memory with name: no-name-898c334e-1231-4322-b546-7c85ee6a5789
[I 2026-08-15 21:57:43,391] Trial 0 finished with value: 0.554738530342075 and parameters: {'C': 0.022441225976441985, 'kernel': 'rbf', 'gamma': 0.00017489421144158533, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.554738530342075.
[I 2026-08-15 21:57:53,743] Trial 1 finished with value: 0.7611249092863348 and parameters: {'C': 75.89235433840862, 'kernel': 'linear', 'class_weight': 'balanced'}. Best is trial 1 with value: 0.7611249092863348.
[I 2026-08-15 21:57:54,888] Trial 2 finished with value: 0.7571556987788333 and parameters: {'C': 30.569113036668178, 'kernel': 'rbf', 'gamma': 0.011148870472313362, 'class_weight': None}. Best is trial 1 with value: 0.7611249092863348.
[I 2026-08-15 21:57:55,860] Trial 3 finished with value: 0.7585145709314981 and parameters: {'C': 2.632384441417451, 'kernel': 'rbf', 'gamma': 0.004694764429246701, 'class_weight': 'balanced


--- ZAKOŃCZONO ---
Najlepsze Accuracy: 0.7751
Najlepsze parametry:
  C: 11.032118783737804
  kernel: rbf
  gamma: 0.005812683832505154
  class_weight: balanced


In [110]:
study.trials_dataframe().sort_values(by='value', ascending=False).head(10)

,number,value,datetime_start,datetime_complete,duration,params_C,params_class_weight,params_gamma,params_kernel,state
79,79,0.775144,2026-08-15 21:58:39.165525,2026-08-15 21:58:39.342871,0 days 00:00:00.177346,11.032119,balanced,0.005813,rbf,COMPLETE
90,90,0.773900,2026-08-15 21:58:42.199097,2026-08-15 21:58:42.524511,0 days 00:00:00.325414,57.485117,balanced,0.002379,rbf,COMPLETE
73,73,0.772884,2026-08-15 21:58:38.061622,2026-08-15 21:58:38.245094,0 days 00:00:00.183472,3.453160,balanced,0.012023,rbf,COMPLETE
85,85,0.772774,2026-08-15 21:58:40.727524,2026-08-15 21:58:41.007769,0 days 00:00:00.280245,36.814803,balanced,0.002398,rbf,COMPLETE
81,81,0.772761,2026-08-15 21:58:39.521020,2026-08-15 21:58:39.741569,0 days 00:00:00.220549,16.250490,balanced,0.007130,rbf,COMPLETE
71,71,0.771180,2026-08-15 21:58:37.689787,2026-08-15 21:58:37.870549,0 days 00:00:00.180762,3.521124,balanced,0.029797,rbf,COMPLETE
86,86,0.770980,2026-08-15 21:58:41.009924,2026-08-15 21:58:41.310581,0 days 00:00:00.300657,45.091312,balanced,0.002237,rbf,COMPLETE
65,65,0.770589,2026-08-15 21:58:36.546455,2026-08-15 21:58:36.730539,0 days 00:00:00.184084,2.502162,balanced,0.016294,rbf,COMPLETE
84,84,0.770573,2026-08-15 21:58:40.398086,2026-08-15 21:58:40.725937,0 days 00:00:00.327851,30.350145,balanced,0.005516,rbf,COMPLETE
82,82,0.770573,2026-08-15 21:58:39.742978,2026-08-15 21:58:40.071911,0 days 00:00:00.328933,31.470455,balanced,0.005442,rbf,COMPLETE


In [111]:
vis.plot_optimization_history(study)

In [112]:
vis.plot_param_importances(study)

In [113]:
vis.plot_parallel_coordinate(study)

# Szukanie hiperparametrów dla MLP

In [114]:
def mlp_search(trial):
  layer_options = {
        '32': (32,),
        '32_16': (32, 16),
        '64': (64,),
        '64_32': (64, 32),
        '128_64': (128, 64),
        '128_64_32': (128, 64, 32)
    }

  layer_name = trial.suggest_categorical('hidden_layer_sizes', list(layer_options.keys()))
  hidden_layer_sizes = layer_options[layer_name]

  activation = trial.suggest_categorical('activation', ['relu', 'tanh'])

  alpha = trial.suggest_float('alpha', 0.00001, 0.1, log=True)
  learning_rate_init = trial.suggest_float('learning_rate_init', 0.0001, 0.1, log=True)

  mlp = MLPClassifier(hidden_layer_sizes=hidden_layer_sizes, activation=activation, alpha=alpha,
                      learning_rate_init=learning_rate_init, max_iter=1000, random_state=42, early_stopping=True)

  pipeline = Pipeline([('scaler', StandardScaler()), ('mlp', mlp)])

  score = cross_val_score(pipeline, X, y, cv=5, scoring='f1').mean()

  return score

study = optuna.create_study(direction='maximize')

study.optimize(mlp_search, n_trials=100)

print("\n--- ZAKOŃCZONO ---")
print(f"Najlepsze Accuracy: {study.best_value:.4f}")
print("Najlepsze parametry:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-08-15 21:58:46,654] A new study created in memory with name: no-name-504e2547-54dc-4940-a7f0-a0910d83b681
[I 2026-08-15 21:58:54,925] Trial 0 finished with value: 0.7460158992907987 and parameters: {'hidden_layer_sizes': '128_64', 'activation': 'tanh', 'alpha': 0.012029848457713642, 'learning_rate_init': 0.0011043990972873772}. Best is trial 0 with value: 0.7460158992907987.
[I 2026-08-15 21:58:56,858] Trial 1 finished with value: 0.7648515599116348 and parameters: {'hidden_layer_sizes': '32', 'activation': 'relu', 'alpha': 0.04043030079856312, 'learning_rate_init': 0.00879907482664625}. Best is trial 1 with value: 0.7648515599116348.
[I 2026-08-15 21:58:59,359] Trial 2 finished with value: 0.7345050974902868 and parameters: {'hidden_layer_sizes': '32_16', 'activation': 'relu', 'alpha': 4.603360722978298e-05, 'learning_rate_init': 0.04099826932001915}. Best is trial 1 with value: 0.7648515599116348.
[I 2026-08-15 21:59:01,933] Trial 3 finished with value: 0.1985090554181814 and


--- ZAKOŃCZONO ---
Najlepsze Accuracy: 0.7764
Najlepsze parametry:
  hidden_layer_sizes: 128_64_32
  activation: relu
  alpha: 4.643799558114539e-05
  learning_rate_init: 0.0016645598304055203


In [115]:
study.trials_dataframe().sort_values(by='value', ascending=False).head(10)

,number,value,datetime_start,datetime_complete,duration,params_activation,params_alpha,params_hidden_layer_sizes,params_learning_rate_init,state
94,94,0.776411,2026-08-15 22:02:36.323532,2026-08-15 22:02:37.962921,0 days 00:00:01.639389,relu,0.000046,128_64_32,0.001665,COMPLETE
89,89,0.776233,2026-08-15 22:02:29.805858,2026-08-15 22:02:31.323679,0 days 00:00:01.517821,relu,0.000143,128_64_32,0.002001,COMPLETE
47,47,0.775761,2026-08-15 22:01:07.087616,2026-08-15 22:01:08.674564,0 days 00:00:01.586948,relu,0.000109,128_64_32,0.001599,COMPLETE
63,63,0.774207,2026-08-15 22:01:39.675018,2026-08-15 22:01:41.173411,0 days 00:00:01.498393,relu,0.000113,128_64_32,0.001603,COMPLETE
26,26,0.773668,2026-08-15 22:00:11.300175,2026-08-15 22:00:13.691278,0 days 00:00:02.391103,tanh,0.000025,128_64_32,0.001766,COMPLETE
69,69,0.773491,2026-08-15 22:01:48.327583,2026-08-15 22:01:50.212969,0 days 00:00:01.885386,relu,0.000433,128_64_32,0.001687,COMPLETE
60,60,0.773033,2026-08-15 22:01:30.596721,2026-08-15 22:01:32.118839,0 days 00:00:01.522118,relu,0.000048,128_64_32,0.001646,COMPLETE
61,61,0.772737,2026-08-15 22:01:32.122829,2026-08-15 22:01:34.238684,0 days 00:00:02.115855,relu,0.000101,128_64_32,0.001669,COMPLETE
81,81,0.772548,2026-08-15 22:02:13.314532,2026-08-15 22:02:14.865226,0 days 00:00:01.550694,relu,0.000736,128_64_32,0.001988,COMPLETE
11,11,0.771122,2026-08-15 21:59:26.438763,2026-08-15 21:59:35.634243,0 days 00:00:09.195480,relu,0.000184,128_64_32,0.001736,COMPLETE


In [116]:
vis.plot_optimization_history(study)

In [117]:
vis.plot_param_importances(study)

In [118]:
vis.plot_parallel_coordinate(study)

# Szukanie hiperparametrów dla KNN

In [119]:
def knn_search(trial):
  n_neighbors = trial.suggest_int('n_neighbors', 1, 50)

  weights = trial.suggest_categorical('weights', ['uniform', 'distance'])

  metric = trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'cosine'])

  knn = KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights, metric=metric)

  pipeline = Pipeline([('scaler', StandardScaler()), ('knn', knn)])

  score = cross_val_score(pipeline, X, y, cv=5, scoring='f1').mean()
  return score

study = optuna.create_study(direction='maximize')

study.optimize(knn_search, n_trials=100)

print("\n--- ZAKOŃCZONO ---")
print(f"Najlepsze Accuracy: {study.best_value:.4f}")
print("Najlepsze parametry:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")


[I 2026-08-15 22:02:52,189] A new study created in memory with name: no-name-c6901235-bd31-475a-8550-684692f7a25d
[I 2026-08-15 22:02:52,305] Trial 0 finished with value: 0.7037843134214061 and parameters: {'n_neighbors': 50, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 0 with value: 0.7037843134214061.
[I 2026-08-15 22:02:52,422] Trial 1 finished with value: 0.719944108362843 and parameters: {'n_neighbors': 14, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 1 with value: 0.719944108362843.
[I 2026-08-15 22:02:52,532] Trial 2 finished with value: 0.7338406025535542 and parameters: {'n_neighbors': 31, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 2 with value: 0.7338406025535542.
[I 2026-08-15 22:02:52,655] Trial 3 finished with value: 0.7078909491152315 and parameters: {'n_neighbors': 30, 'weights': 'uniform', 'metric': 'manhattan'}. Best is trial 2 with value: 0.7338406025535542.
[I 2026-08-15 22:02:52,802] Trial 4 finished with value: 0.71930108921493


--- ZAKOŃCZONO ---
Najlepsze Accuracy: 0.7525
Najlepsze parametry:
  n_neighbors: 31
  weights: distance
  metric: euclidean


In [120]:
study.trials_dataframe().sort_values(by='value', ascending=False).head(10)

,number,value,datetime_start,datetime_complete,duration,params_metric,params_n_neighbors,params_weights,state
20,20,0.752546,2026-08-15 22:02:56.483588,2026-08-15 22:02:56.596277,0 days 00:00:00.112689,euclidean,31,distance,COMPLETE
51,51,0.752546,2026-08-15 22:03:00.439492,2026-08-15 22:03:00.522828,0 days 00:00:00.083336,euclidean,31,distance,COMPLETE
50,50,0.752546,2026-08-15 22:03:00.361539,2026-08-15 22:03:00.438249,0 days 00:00:00.076710,euclidean,31,distance,COMPLETE
38,38,0.752546,2026-08-15 22:02:59.044353,2026-08-15 22:02:59.169720,0 days 00:00:00.125367,euclidean,31,distance,COMPLETE
55,55,0.752049,2026-08-15 22:03:00.774706,2026-08-15 22:03:00.854054,0 days 00:00:00.079348,euclidean,21,distance,COMPLETE
57,57,0.752049,2026-08-15 22:03:00.942663,2026-08-15 22:03:01.019505,0 days 00:00:00.076842,euclidean,21,distance,COMPLETE
59,59,0.752049,2026-08-15 22:03:01.097607,2026-08-15 22:03:01.175834,0 days 00:00:00.078227,euclidean,21,distance,COMPLETE
61,61,0.752049,2026-08-15 22:03:01.289807,2026-08-15 22:03:01.387505,0 days 00:00:00.097698,euclidean,21,distance,COMPLETE
71,71,0.752049,2026-08-15 22:03:02.106512,2026-08-15 22:03:02.187556,0 days 00:00:00.081044,euclidean,21,distance,COMPLETE
81,81,0.752049,2026-08-15 22:03:02.958721,2026-08-15 22:03:03.034394,0 days 00:00:00.075673,euclidean,21,distance,COMPLETE


In [121]:
vis.plot_optimization_history(study)

In [122]:
vis.plot_param_importances(study)

In [123]:
vis.plot_parallel_coordinate(study)

# Moje szanse na tytaniku 💀

In [124]:
print(titanic_df.columns)
titanic_df.head()

Index(['Survived', 'Pclass', 'Sex', 'Age', 'Fare', 'FamilySize', 'IsAlone',
       'Embarked_Q', 'Embarked_S', 'Cabin_B', 'Cabin_C', 'Cabin_D', 'Cabin_E',
       'Cabin_F', 'Cabin_G', 'Cabin_T', 'Cabin_Unknown', 'Title_Miss',
       'Title_Mr', 'Title_Mrs', 'Title_Rare'],
      dtype='object')


,Survived,Pclass,Sex,Age,Fare,FamilySize,IsAlone,Embarked_Q,Embarked_S,Cabin_B,...,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Cabin_Unknown,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
PassengerId,,,,,,,,,,,,,,,,,,,,,
1,0,3,0,22.0,7.2500,2,0,0,1,0,...,0,0,0,0,0,1,0,1,0,0
2,1,1,1,38.0,71.2833,2,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,1,3,1,26.0,7.9250,1,1,0,1,0,...,0,0,0,0,0,1,1,0,0,0
4,1,1,1,35.0,53.1000,2,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
5,0,3,0,35.0,8.0500,1,1,0,1,0,...,0,0,0,0,0,1,0,1,0,0


In [125]:
ja = pd.DataFrame([{
    'Pclass': 1,
    'Sex': 0,
    'Age': 21.0,
    'Fare': 53.0,
    'FamilySize': 2,
    'IsAlone': 0,
    'Embarked_Q': 1,
    'Embarked_S': 0,
    'Cabin_B': 0,
    'Cabin_C': 0,
    'Cabin_D': 0,
    'Cabin_E': 0,
    'Cabin_F': 0,
    'Cabin_G': 0,
    'Cabin_T': 0,
    'Cabin_Unknown': 0,
    'Title_Miss': 0,
    'Title_Mr': 1,
    'Title_Mrs': 0,
    'Title_Rare': 0
}])

# użyje dobrych parametrów
  # * wartości mogą się różnić bo raczej odpale cały plik jak skończę
svm = SVC(kernel='rbf', C=7.925165, gamma=0.012747, class_weight=None)
svm.fit(X_train_scaled, y_train)

mlp = MLPClassifier(max_iter=1000, random_state=42, hidden_layer_sizes=(128, 64, 32), alpha=0.000312, learning_rate_init=0.001695, early_stopping=True)
mlp.fit(X_train_scaled, y_train)

knn = KNeighborsClassifier(n_neighbors=31, metric='euclidean', weights='distance')
knn.fit(X_train_scaled, y_train)

ja_scaled = scaler.transform(ja)

svm_pred = svm.predict(ja_scaled)
mlp_pred = mlp.predict(ja_scaled)
knn_pred = knn.predict(ja_scaled)

for pred in [svm_pred, mlp_pred, knn_pred]:
  if pred[0] == 1:
    print('Żyje!!!')
  else:
    print('GG')

GG
Żyje!!!
GG


# Wnioski
* niestety wygląda na to, że nie przeżyłbym na tytaniku, a przynajmniej 2 na 3 modele tak zdecydowały (mimo, że dałem sobie 1 klasę)
* wszystkie 3 modele pokonały model bazowy
* wszystkie 3 modele osiągnęły bardzo podobny poziom
* poprawność przewidywania jest znacząco zależna od tego jakie dane wylosują się jako treningowe